In [1]:
import pandas as pd
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

d:\git repos\AI_optimization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("query,use_memory.csv")
queries = df['query'].tolist()
labels = df["use_memory"].map({"No":0, "Yes":1}).tolist()

In [3]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    queries, labels, test_size=0.2, random_state=2026
)

In [4]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

In [6]:
class QueryDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    
train_dataset = QueryDataset(train_encodings, train_labels)
test_dataset = QueryDataset(test_encodings, test_labels)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2  
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}


In [12]:
training_args = TrainingArguments(
    output_dir="./distilbert-query-classifier",
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs",
    logging_steps=10,
    learning_rate=5e-5,
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


C:\Users\VISHNU\AppData\Local\Temp\ipykernel_21428\2147312783.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
trainer.train()

d:\git repos\AI_optimization\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.546600
20,0.118000
30,0.014700
40,0.029200
50,0.013700
60,0.001900
70,0.001800
80,0.001200
90,0.001000
100,0.000900


TrainOutput(global_step=320, training_loss=0.023064583240193316, metrics={'train_runtime': 139.6567, 'train_samples_per_second': 36.088, 'train_steps_per_second': 2.291, 'total_flos': 18255663377280.0, 'train_loss': 0.023064583240193316, 'epoch': 20.0})

In [15]:
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)

d:\git repos\AI_optimization\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluation results: {'eval_loss': 0.00020137621322646737, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_runtime': 0.4646, 'eval_samples_per_second': 137.755, 'eval_steps_per_second': 8.61, 'epoch': 20.0}


In [16]:
model.save_pretrained("./distilbert-memory-classifier")
tokenizer.save_pretrained("./distilbert-memory-classifier")

('./distilbert-memory-classifier\\tokenizer_config.json',
 './distilbert-memory-classifier\\special_tokens_map.json',
 './distilbert-memory-classifier\\vocab.txt',
 './distilbert-memory-classifier\\added_tokens.json',
 './distilbert-memory-classifier\\tokenizer.json')